# 01 · Carga de tickets a tablas Delta (Banco Altiplano)

**Qué hace (RF-23, S6):** lee los CSV sintéticos de `Files/raw/`, los limpia y tipa, revisa su
calidad y escribe las tablas Delta `tickets` y `sucursales` en el lakehouse. El SQL analytics
endpoint del lakehouse las expone (solo lectura) para las 10 consultas de
`integrations/fabric/sql/`. Todos los datos son 100% sintéticos.

**Antes de correrlo:**

1. Importa este `.ipynb` en tu workspace con la opción de importar notebook (Fabric reconoce
   archivos `.ipynb` estándar).
2. En el Lakehouse explorer agrega tu lakehouse y fíjalo como predeterminado (pin). Las rutas
   relativas `Files/...` y `saveAsTable` usan ese lakehouse. Si lo cambias, reinicia la sesión.
3. Sube `tickets.csv` y `sucursales.csv` (salen de `python main.py generate-data`, carpeta
   `data/raw/`) a la carpeta `Files/raw/` del lakehouse.

> ⚠ No verificado en Spark: en el repo solo se valida el JSON del notebook y la sintaxis
> Python de cada celda. La ejecución real se valida en Fabric en la tarea T29.

## 1. Configuración

Rutas relativas al lakehouse predeterminado y lista de columnas en el mismo orden que escribe
el generador (`src/ingestion/ticket_generator.py`). En Fabric la sesión de Spark ya existe:
`SparkSession.builder.getOrCreate()` devuelve esa misma sesión.

In [ ]:
from datetime import datetime

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.getOrCreate()

RAW_DIR = "Files/raw"
TICKETS_CSV = f"{RAW_DIR}/tickets.csv"
SUCURSALES_CSV = f"{RAW_DIR}/sucursales.csv"
TIMESTAMP_FORMAT = "yyyy-MM-dd HH:mm:ss"
PERIOD_START = datetime(2025, 11, 1)
PERIOD_END = datetime(2026, 11, 1)  # exclusive: the synthetic period ends on 2026-10-31

TICKET_COLUMNS = [
    "ticket_id",
    "fecha_apertura",
    "fecha_cierre",
    "categoria",
    "subcategoria",
    "prioridad",
    "sucursal",
    "area",
    "canal",
    "sla_horas",
    "sla_cumplido",
    "horas_resolucion",
    "anio_mes",
    "solucion",
    "es_repetitivo",
    "idioma",
    "estado",
]
SUCURSAL_COLUMNS = ["sucursal", "ciudad", "estado_mx", "region", "es_foranea"]

## 2. Lectura con esquema explícito

La capa cruda se lee **con un esquema explícito, todo como texto**, en lugar de `inferSchema`
(que recorre los datos y puede adivinar mal un tipo). Así primero se limpian los espacios y
después cada columna se convierte de forma controlada.

- `mode = FAILFAST`: falla si una fila viene mal formada.
- `escape = "`: acepta comillas dobladas, como las escribe el módulo `csv` de Python.

In [ ]:
def raw_schema(columns: list[str]) -> T.StructType:
    return T.StructType([T.StructField(name, T.StringType(), True) for name in columns])


def read_raw(path: str, columns: list[str]) -> DataFrame:
    return (
        spark.read.format("csv")
        .option("header", True)
        .option("encoding", "UTF-8")
        .option("escape", '"')
        .option("mode", "FAILFAST")
        .schema(raw_schema(columns))
        .load(path)
    )


tickets_raw = read_raw(TICKETS_CSV, TICKET_COLUMNS)
sucursales_raw = read_raw(SUCURSALES_CSV, SUCURSAL_COLUMNS)
print(f"tickets_raw: {tickets_raw.count()} filas | sucursales_raw: {sucursales_raw.count()} filas")

## 3. Limpieza y tipos

- Se quitan espacios al inicio y al final, y los textos vacíos pasan a `NULL` (por ejemplo,
  `fecha_cierre` de un ticket abierto).
- Fechas a `timestamp` (el SQL analytics endpoint las expone como `datetime2`).
- Banderas `sla_cumplido` y `es_repetitivo` (y `es_foranea`) como enteros 0/1, igual que en
  SQLite, para que `SUM(bandera)` funcione igual en las 10 consultas.
- `horas_resolucion` como `double`; `anio_mes` se queda como texto `YYYY-MM`.
- Duplicados: se conserva una fila por `ticket_id` (y por `sucursal`).

In [ ]:
def trim_and_nullify(df: DataFrame) -> DataFrame:
    """Trim every string column and turn empty strings into NULL."""
    return df.select(
        [
            F.when(F.trim(F.col(name)) == "", None).otherwise(F.trim(F.col(name))).alias(name)
            for name in df.columns
        ]
    )


tickets_trimmed = trim_and_nullify(tickets_raw)
sucursales_trimmed = trim_and_nullify(sucursales_raw)

tickets = (
    tickets_trimmed.withColumn("fecha_apertura", F.to_timestamp("fecha_apertura", TIMESTAMP_FORMAT))
    .withColumn("fecha_cierre", F.to_timestamp("fecha_cierre", TIMESTAMP_FORMAT))
    .withColumn("sla_horas", F.col("sla_horas").cast("int"))
    .withColumn("sla_cumplido", F.col("sla_cumplido").cast("int"))
    .withColumn("horas_resolucion", F.col("horas_resolucion").cast("double"))
    .withColumn("es_repetitivo", F.col("es_repetitivo").cast("int"))
    .dropDuplicates(["ticket_id"])
    .select(TICKET_COLUMNS)
)
sucursales = (
    sucursales_trimmed.withColumn("es_foranea", F.col("es_foranea").cast("int"))
    .dropDuplicates(["sucursal"])
    .select(SUCURSAL_COLUMNS)
)
tickets.printSchema()

## 4. Chequeos de calidad

Cada chequeo imprime `OK` o detiene el notebook con un error: es mejor no cargar que cargar
datos malos. Se revisan los encabezados, los conteos y duplicados, los nulos en las claves,
las conversiones de tipo, el rango de fechas, las banderas 0/1 y que toda sucursal exista.

Si Spark trabaja en modo ANSI, una conversión inválida falla ya en el paso anterior; si no,
el valor queda en `NULL` y lo detecta `invalid_casts`.

In [ ]:
def check(condition: bool, message: str) -> None:
    if not condition:
        raise ValueError(f"Quality check failed: {message}")
    print(f"OK  {message}")


def invalid_casts(raw: DataFrame, clean: DataFrame, key: str, columns: list[str]) -> int:
    """Rows where a non-empty raw value became NULL after the type conversion."""
    lost = F.lit(False)
    for name in columns:
        lost = lost | (F.col(f"r.{name}").isNotNull() & F.col(f"c.{name}").isNull())
    return raw.alias("r").join(clean.alias("c"), on=key).filter(lost).count()


def header_of(path: str) -> list[str]:
    return spark.read.text(path).first()[0].split(",")


check(header_of(TICKETS_CSV) == TICKET_COLUMNS, "tickets.csv trae las 17 columnas en orden")
check(header_of(SUCURSALES_CSV) == SUCURSAL_COLUMNS, "sucursales.csv trae sus 5 columnas")

n_raw, n_tickets = tickets_raw.count(), tickets.count()
n_sucursales = sucursales.count()
print(f"tickets: {n_raw} leídos, {n_tickets} tras limpiar ({n_raw - n_tickets} duplicados)")
check(
    n_tickets > 0 and n_sucursales > 0,
    f"hay datos ({n_tickets} tickets, {n_sucursales} sucursales)",
)

for key in ["ticket_id", "fecha_apertura", "sucursal", "prioridad", "estado"]:
    check(tickets.filter(F.col(key).isNull()).count() == 0, f"sin nulos en tickets.{key}")
check(
    sucursales.filter(F.col("sucursal").isNull()).count() == 0, "sin nulos en sucursales.sucursal"
)

typed = [
    "fecha_apertura",
    "fecha_cierre",
    "sla_horas",
    "sla_cumplido",
    "horas_resolucion",
    "es_repetitivo",
]
check(
    invalid_casts(tickets_trimmed, tickets, "ticket_id", typed) == 0,
    "fechas y números convertidos sin perder valores",
)
check(
    invalid_casts(sucursales_trimmed, sucursales, "sucursal", ["es_foranea"]) == 0,
    "es_foranea convertido sin perder valores",
)

bounds = tickets.agg(
    F.min("fecha_apertura").alias("min"), F.max("fecha_apertura").alias("max")
).first()
print(f"Rango de fecha_apertura: {bounds['min']} .. {bounds['max']}")
check(
    bounds["min"] >= PERIOD_START and bounds["max"] < PERIOD_END,
    "fecha_apertura dentro de 2025-11-01 .. 2026-10-31",
)
check(
    tickets.filter(F.col("fecha_cierre") < F.col("fecha_apertura")).count() == 0,
    "fecha_cierre nunca es anterior a fecha_apertura",
)
check(
    tickets.filter(F.col("anio_mes") != F.date_format("fecha_apertura", "yyyy-MM")).count() == 0,
    "anio_mes coincide con fecha_apertura",
)

for df, flag in [(tickets, "sla_cumplido"), (tickets, "es_repetitivo"), (sucursales, "es_foranea")]:
    check(
        df.filter(F.col(flag).isNotNull() & ~F.col(flag).isin(0, 1)).count() == 0,
        f"{flag} solo vale 0/1",
    )

orphans = tickets.join(sucursales, on="sucursal", how="left_anti").count()
check(orphans == 0, "toda sucursal de tickets existe en sucursales")

## 5. Escritura a tablas Delta

`mode("overwrite")` + `saveAsTable` crea (o reemplaza) tablas administradas en la sección
**Tables** del lakehouse predeterminado. Volver a correr el notebook deja el mismo resultado.

In [ ]:
tickets.write.mode("overwrite").format("delta").saveAsTable("tickets")
sucursales.write.mode("overwrite").format("delta").saveAsTable("sucursales")
print("Tablas Delta escritas: tickets, sucursales")

## 6. Verificación

Los conteos leídos de las tablas Delta deben coincidir con los de la limpieza. Después, en el
SQL analytics endpoint del lakehouse, `SELECT COUNT(*) FROM tickets` debe dar lo mismo y ya se
pueden correr las 10 consultas de `integrations/fabric/sql/` (T29).

In [ ]:
for table, expected in [("tickets", n_tickets), ("sucursales", n_sucursales)]:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {table}").first()["n"]
    check(n == expected, f"{table}: {n} filas en Delta")

summary = (
    "SELECT COUNT(*) AS n_tickets, MIN(fecha_apertura) AS desde, MAX(fecha_apertura) AS hasta "
    "FROM tickets"
)
spark.sql(summary).show()